# Web Scraping Tutorial

This notebook shows how I scraped data from the Production Inspiration website. The goal is to understand the process so you can apply it to other websites.

**What we're doing:** Automatically collecting information from web pages and saving it to a CSV file.

**Tools:** Python, Selenium (controls browser), Pandas (organizes data)

## Step 1: Understanding the Website

First, I visited the website to see how it's organized.

**Website:** https://www.productioninspiration.com

The site has different "effects" (like Absorb, Heat, Mix) and each effect has examples for different states (Solid, Liquid, Gas, Field).

**Example URL:** https://www.productioninspiration.com/1/absorb/solid

Each page shows multiple items with:
- **Title** (in an `<h3>` tag)
- **Description** (in a `<p>` tag)
- **Example** (optional, in a `<small>` tag)

![Homepage](screenshot_1_homepage.png)


![Sample Page](screenshot_2_sample_page.png)

## Step 2: Inspecting the HTML

To extract data, I need to know the HTML structure. I used browser Developer Tools:
1. Right-click on an item card
2. Select "Inspect"
3. Look at the HTML structure

**Key findings:**
- Each item is in a `<div class="col-md-6 item">`
- Title is in `<h3>`
- Description is in `<p>` (but not the one with class "item-example")
- Example is in `<div class="item-example"><small>`


![Developer Tools](screenshot_03_developer_tools.png)

By selecting an element on the page and clicking it, you can locate its position within the HTML structure.

![Developer Tools](screenshot_03_2_developer_tools.png)

## Step 3: Setting Up

I'll use just one effect ("Absorb") as an example. In the real project, I had many more.

In [1]:
# URLs for the "Absorb" effect
urls = {
    "Solid": "https://www.productioninspiration.com/1/absorb/solid",
    "Liquid": "https://www.productioninspiration.com/1/absorb/liquid",
    "Gas": "https://www.productioninspiration.com/1/absorb/gas",
    "Field": "https://www.productioninspiration.com/1/absorb/field"
}

## Step 4: Import Libraries

Import the tools we need.

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

## Step 5: Open Browser

Selenium opens a Chrome browser that we can control with code.

In [3]:
driver = webdriver.Chrome()
print("Browser opened")

Browser opened



![Browser Opening](screenshot_04_browser_opening.png)

## Step 6: The Scraping Process

Here's the main loop that does the work:
1. Visit each URL
2. Wait for items to load
3. Find all items on the page
4. Extract data from each item
5. Save to a list

In [4]:
# Store results here
results = []

# Loop through each URL
for state, url in urls.items():
    print(f"Visiting: {state}...")

    # Go to the page
    driver.get(url)

    # Wait for items to appear (important for dynamic sites)
    try:
        WebDriverWait(driver, 4).until(
            EC.presence_of_element_located((By.CLASS_NAME, "item"))
        )
    except:
        print(f"  Timeout on {url}")
        continue

    # Find all items on the page
    items = driver.find_elements(By.CLASS_NAME, "item")
    print(f"  Found {len(items)} items")

    # Extract data from each item
    for item in items:
        try:
            # Get title
            title = item.find_element(By.TAG_NAME, "h3").text

            # Get description (p tag that's not in item-example)
            description = item.find_element(
                By.XPATH, ".//p[not(contains(@class, 'item-example'))]"
            ).text

            # Get example if it exists
            try:
                example = item.find_element(
                    By.CLASS_NAME, "item-example"
                ).find_element(By.TAG_NAME, "small").text
            except:
                example = None

            # Save the data
            results.append({
                "Effect": "Absorb",
                "State": state,
                "Title": title,
                "URL": url,
                "Description": description,
                "Example": example
            })
        except Exception as e:
            print(f"  Error: {e}")
            continue

print(f"\nDone! Collected {len(results)} items")

Visiting: Solid...
  Found 1 items
Visiting: Liquid...
  Found 5 items
Visiting: Gas...
  Found 2 items
Visiting: Field...
  Found 6 items

Done! Collected 14 items


## Step 7: Close Browser

Always close the browser when done

In [5]:
driver.quit()
print("Browser closed")

Browser closed


## Step 8: Save to CSV

Convert the list to a DataFrame and save as CSV.

In [6]:
# Create DataFrame
df = pd.DataFrame(results)

# Save to CSV
df.to_csv('scraped_data.csv', index=False)

# save to excel 
df.to_excel('scraped_data.xlsx', index=False)

print(f"Saved {len(df)} items to scraped_data.csv")

Saved 14 items to scraped_data.csv


## Step 9: View Results

Let's see what we collected.

In [7]:
# Show first few rows
df.head()

,Effect,State,Title,URL,Description,Example
0,Absorb,Solid,Shape_memory effect,https://www.productioninspiration.com/1/absorb...,Shape Memory Effect is a reversible solid stat...,Example: Shape memory alloys(SMA) can be used ...
1,Absorb,Liquid,Capillary Action,https://www.productioninspiration.com/1/absorb...,Capillary Action is a physical phenomenon of l...,Example: When a piece of chalk is dipped in a ...
2,Absorb,Liquid,Chemisorption,https://www.productioninspiration.com/1/absorb...,"Chemisorption or Chemical Adsorption, involves...",Example: Animation illustrates the concept.
3,Absorb,Liquid,Hydrogenation,https://www.productioninspiration.com/1/absorb...,Hydrogenation is the chemical process involvin...,Example: Hydrogenation of oil.
4,Absorb,Liquid,Exothermic Reactions,https://www.productioninspiration.com/1/absorb...,Exothermic Reaction is a chemical reaction tha...,None


In [8]:
# Count items by state
print("Items by state:")
print(df['State'].value_counts())

Items by state:
State
Field     6
Liquid    5
Gas       2
Solid     1
Name: count, dtype: int64


## Key Concepts Explained

**Why Selenium?**
- Many websites load content with JavaScript
- Selenium controls a real browser, so it can handle dynamic content
- Regular HTTP requests (like `requests` library) can't execute JavaScript

**Why wait for elements?**
- Pages don't load instantly
- `WebDriverWait` ensures elements exist before we try to extract them
- Prevents errors from trying to find elements that aren't there yet

**Why try-except?**
- Some items might be missing data
- Some pages might not load
- Try-except lets the script continue even if one item fails

**XPATH selector:** `.//p[not(contains(@class, 'item-example'))]`
- Finds paragraph tags that don't have class "item-example"
- This gets the description, not the example text

## How to Apply This to Other Websites

The process is always the same:

1. **Visit the website** - Understand its structure
2. **Inspect HTML** - Use Developer Tools to find the elements you need
3. **Identify selectors** - What class names, tags, or IDs to look for
4. **Write the loop** - Visit pages, wait for content, extract data
5. **Handle errors** - Use try-except to keep going if something fails
6. **Save data** - Export to CSV, JSON, or database

**Tips:**
- Start with one page to test your code
- Add delays between requests if needed (be respectful)
- Check the website's terms of service
- Save progress regularly for large jobs

## Summary

**What we did:**
- Used Selenium to control a browser
- Visited multiple pages automatically
- Extracted structured data (title, description, example)
- Saved everything to a CSV file

**Result:** 14 items collected from the Production Inspiration website

The process works for any website - just adapt the selectors and URL structure to match your target site.

## Legal and Ethical Considerations

**⚠️ IMPORTANT:** Before scraping any website, you must check the following:

### Legal Requirements

**1. Check robots.txt**
- Visit: `https://website.com/robots.txt`
- This file tells you what the website allows/disallows
- If scraping is disallowed, don't do it
- Example: `https://www.productioninspiration.com/robots.txt`

**2. Read Terms of Service**
- Look for a "Terms of Service" or "Terms of Use" page
- Many websites explicitly prohibit scraping in their terms
- Violating terms can lead to legal action
- If unsure, contact the website owner for permission

**3. Copyright and Data Ownership**
- Website content is usually copyrighted
- You can't republish scraped content without permission
- Personal data (names, emails, etc.) may be protected by privacy laws (GDPR, etc.)
- Check what you're allowed to do with the data

**4. Rate Limiting and Server Load**
- Don't overload servers with too many requests
- Add delays between requests (e.g., `time.sleep(2)`)
- Be respectful of the website's resources
- Too many requests can get your IP blocked

### Ethical Guidelines

**1. Be Respectful**
- Scrape during off-peak hours if possible
- Don't scrape more than you need
- Respect the website's bandwidth and resources

**2. Identify Yourself**
- Use a proper User-Agent header
- Some sites may block unknown bots
- Be transparent about what you're doing

**3. Data Usage**
- Only use scraped data for legitimate purposes
- Don't resell data without permission
- Respect privacy - don't scrape personal information unnecessarily

**4. When in Doubt, Ask**
- Contact the website owner for permission
- Many sites have APIs that are better than scraping
- APIs are usually faster, more reliable, and legal

### Best Practices Checklist

Before scraping, ask yourself:
- ☐ Did I check robots.txt?
- ☐ Did I read the Terms of Service?
- ☐ Am I respecting rate limits?
- ☐ Is there an API I could use instead?
- ☐ Do I have permission to use this data?
- ☐ Am I being respectful of the server?

**Remember:** Just because you *can* scrape a website doesn't mean you *should*. Always prioritize legal compliance and ethical behavior.